# Multimodal integration of 10x Xenium and Imaging mass cytometry (IMC) visualization in Rakaia

Thie notebook completes the multimodal transformation and integration to create the inputs needed for Extended Data Figure 1. It assumes that the user has already run **process_raw_xenium.ipynb** to generate the necessary 10x Xenium input files.  

In [2]:
import numpy as np
import pandas as pd
from skimage.transform import warp
from tifffile import imread, imwrite, TiffFile
from numpy.linalg import inv
from rakaia.parsers.spatial import (spatial_canvas_dimensions, spatial_grid_single_marker)
import rasterio
from rasterio.features import rasterize
import warnings
import anndata as ad

warnings.filterwarnings('ignore')

**Before starting**: This notebook assumes that you have 

1. performed an alignment of the 10x Xenium DAPI IF morphology image to the IMC ROI `imc_roi_2.tiff` using [twocan](https://github.com/camlab-bioml/twocan). Information on performing the alignment can be found [here](https://gist.github.com/harrig12/aa2f48220051eed9e28bbcb3567229b4), and
2. Run through `process_raw_xenium.ipynb`

To run this notebook, the following data are required in the same directory as this notebook:
- the IMC ROI in tiff format named `imc_roi_2.tiff`
- The cell-level transcript expression for the 10x Xenium slide in h5ad format named `xenium_expression.h5ad`, generated from `process_raw_xenium.ipynb`
- The Xenium cell segmentation mask named `xenium_mask.tiff`, generated from `process_raw_xenium.ipynb`
- The output affine transformation matrix in CSV format from [twocan](https://github.com/camlab-bioml/twocan) named `affine_twocan.csv`
- The directory for the Xenium morphology images, named as `morphology_focus`

First, we will import the Xenium cell expression, and compute some visualisation offsets for Rakaia. 


In [3]:
expr = ad.read_h5ad("./xenium_expression.h5ad")

grid_width, grid_height, x_min, y_min = spatial_canvas_dimensions(expr)

# # apply the offset to the transform to take into account the trimming of dead space in rakaia
T_offset_inv = np.array([
    [0, 0, -x_min],
    [0, 0, -y_min],
    [0, 0, 0]
])

Next, we read in the raw IMC data and the Xenium DAPI morphology image, and transform the DAPI signal into the IMC resolution to check the concordance of the signal overlap. 

In [4]:
with TiffFile("imc_roi_2.tiff") as tif:
    imc_stack = tif.asarray()

tform1 = np.array(pd.read_csv("twocan_affine.csv", header=None))

with TiffFile("morphology_focus/morphology_focus_0000.ome.tif") as tif:
    if_dapi = tif.pages[0].asarray().astype(np.float32)
    dapi_transform = warp(
        image=if_dapi,
        inverse_map=np.vstack([(tform1 / 0.2125), [0,0,1]]),
        output_shape=(imc_stack.shape[1], imc_stack.shape[2]),
        order=1,
        mode='constant',
        cval=0.0,
        preserve_range=True
    ).astype(np.uint32)

We then define a subset of Xenium markers to append to the IMC stack, and rasterise these markers into "dense" numpy arrays that can be visualised in Rakaia in the same fielf of view (FOV) as the IMC. 

In [5]:
tform1 = np.vstack([tform1, [0,0,1]])
tform1 = tform1 + T_offset_inv

genes_use = ['CXCL12', 'CXCL2', 'GATA3', 'TP63', 'TNFSF10']

xenium_transformed_stacks = []

for gene in genes_use:
    raster = spatial_grid_single_marker(expr, gene, 5, True, False)
    raster_transform = warp(
        image=raster,
        inverse_map=tform1,
        output_shape=(imc_stack.shape[1], imc_stack.shape[2]),
        order=0,
        mode='constant',
        cval=0.0,
        preserve_range=True
    ).astype(np.uint32)
    xenium_transformed_stacks.append(raster_transform)

xenium_transformed_stacks.append(dapi_transform)

imwrite("imc_with_xenium_markers.tiff",
    np.concatenate([imc_stack, np.stack(xenium_transformed_stacks, axis=0)], axis=0).astype(np.float32),
    photometric="minisblack")

Lastly, we will  the Xenium single cell segmentation mask into the IMC resolution, so that we can visualize the Xenium single cells projected over both the IMC and Xenium multimodal images. 

In [6]:
with TiffFile("xenium_mask.tiff") as tif:
    xenium_mask = tif.pages[0].asarray().astype(np.uint32)
    mask_transform = warp(
        image=xenium_mask,
        inverse_map=tform1,
        output_shape=(imc_stack.shape[1], imc_stack.shape[2]),
        order=0,
        mode='constant',
        cval=0.0,
        preserve_range=True
    ).astype(np.uint32)

imwrite("xenium_mask_transformed.tiff", mask_transform.astype(np.uint32),
    photometric="minisblack")